# PR. 3 – Spread Locator
### Mathematics & Advanced Statistics
Statistical Distribution Analysis


## 1. Import Libraries


In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Read the supplied dataset
DATA_FILE = "spread_locator_dataset.xlsx"
df = pd.read_excel(DATA_FILE)

print("Dataset shape:", df.shape)
display(df.head())


## 2. Load and Inspect Dataset


In [ ]:
# Basic inspection
print("Columns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isna().sum())

print("\nData types:")
print(df.dtypes)

print("\nTransaction status:")
print(df["transaction_status"].value_counts())

print("\nTransaction amount summary:")
display(df["transaction_amount"].describe())


## 3. Bernoulli Distribution


In [ ]:
# Bernoulli fit using transaction status
df["success"] = (df["transaction_status"] == "Success").astype(int)

p_success = df["success"].mean()

print(f"Estimated Bernoulli p (Success): {p_success:.4f}")
print(f"Observed success rate: {p_success*100:.2f}%")

# Compare observed and theoretical probabilities
bernoulli_table = pd.DataFrame({
    "Outcome": ["Fail (0)", "Success (1)"],
    "Observed Probability": [1-p_success, p_success]
})
display(bernoulli_table)


## 4. Binomial Distribution


In [ ]:
# Binomial fit to transaction_count
counts = df["transaction_count"].astype(int)

n_binom = int(counts.max())
p_binom = counts.mean() / n_binom
p_binom = min(max(p_binom, 0), 1)

print(f"Binomial n: {n_binom}")
print(f"Estimated p: {p_binom:.4f}")
print(f"Observed mean weekly count: {counts.mean():.4f}")
print(f"Binomial expected count (n*p): {n_binom*p_binom:.4f}")

# Observed vs fitted PMF
x = np.arange(0, n_binom + 1)
observed_pmf = counts.value_counts(normalize=True).reindex(x, fill_value=0)
fitted_pmf = stats.binom.pmf(x, n_binom, p_binom)

binom_compare = pd.DataFrame({
    "Count": x,
    "Observed Probability": observed_pmf.values,
    "Fitted Binomial PMF": fitted_pmf
})
display(binom_compare)


## 5. Poisson Distribution


In [ ]:
plt.figure(figsize=(8,5))
width = 0.38
plt.bar(x - width/2, observed_pmf.values, width=width, label="Observed")
plt.bar(x + width/2, fitted_pmf, width=width, label="Binomial fit")
plt.xlabel("Weekly Transaction Count")
plt.ylabel("Probability")
plt.title("Observed vs Binomial Distribution")
plt.legend()
plt.tight_layout()
plt.show()


## 6. Log-Normal Distribution


In [ ]:
daily_transactions = df.groupby("transaction_date").size()
lambda_poisson = daily_transactions.mean()

print(f"Number of days: {len(daily_transactions)}")
print(f"Estimated Poisson lambda: {lambda_poisson:.4f}")

daily_x = np.arange(daily_transactions.min(), daily_transactions.max()+1)
poisson_pmf = stats.poisson.pmf(daily_x, lambda_poisson)

poisson_table = pd.DataFrame({
    "Daily Count": daily_x,
    "Observed Frequency": daily_transactions.value_counts().reindex(daily_x, fill_value=0).values,
    "Poisson PMF": poisson_pmf
})
display(poisson_table)

plt.figure(figsize=(8,5))
plt.bar(daily_x, poisson_pmf, alpha=0.7)
plt.xlabel("Transactions per Day")
plt.ylabel("Poisson Probability")
plt.title("Poisson Distribution Fit – Daily Transactions")
plt.tight_layout()
plt.show()


## 7. Power Law Distribution


In [ ]:
amounts = df["transaction_amount"].astype(float)

# Log-normal fit
ln_shape, ln_loc, ln_scale = stats.lognorm.fit(amounts, floc=0)
ln_loglik = np.sum(stats.lognorm.logpdf(amounts, ln_shape, loc=ln_loc, scale=ln_scale))
ln_aic = 2*2 - 2*ln_loglik
ln_ks = stats.kstest(amounts, "lognorm", args=(ln_shape, ln_loc, ln_scale))

# Pareto / power-law fit
pa_shape, pa_loc, pa_scale = stats.pareto.fit(amounts, floc=0)
pa_loglik = np.sum(stats.pareto.logpdf(amounts, pa_shape, loc=pa_loc, scale=pa_scale))
pa_aic = 2*2 - 2*pa_loglik
pa_ks = stats.kstest(amounts, "pareto", args=(pa_shape, pa_loc, pa_scale))

fit_table = pd.DataFrame({
    "Model": ["Log-Normal", "Power Law / Pareto"],
    "K-S Statistic": [ln_ks.statistic, pa_ks.statistic],
    "K-S p-value": [ln_ks.pvalue, pa_ks.pvalue],
    "AIC": [ln_aic, pa_aic]
})
display(fit_table)

print(f"Log-normal shape = {ln_shape:.4f}, scale = {ln_scale:.2f}")
print(f"Pareto shape = {pa_shape:.4f}, scale = {pa_scale:.2f}")


## 8. Q-Q Plot


In [ ]:
plt.figure(figsize=(7,6))
stats.probplot(amounts, dist="norm", plot=plt)
plt.title("Q-Q Plot of Transaction Amounts")
plt.tight_layout()
plt.show()

shapiro_stat, shapiro_p = stats.shapiro(amounts)
print(f"Shapiro-Wilk statistic: {shapiro_stat:.4f}")
print(f"Shapiro-Wilk p-value: {shapiro_p:.6g}")


## 9. Box-Cox Transformation


In [ ]:
boxcox_values, boxcox_lambda = stats.boxcox(amounts)

print(f"Estimated Box-Cox lambda: {boxcox_lambda:.4f}")
print(f"Original skewness: {amounts.skew():.4f}")
print(f"Transformed skewness: {pd.Series(boxcox_values).skew():.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12,5))

axes[0].hist(amounts, bins=25)
axes[0].set_title("Original Transaction Amounts")
axes[0].set_xlabel("Transaction Amount")
axes[0].set_ylabel("Frequency")

axes[1].hist(boxcox_values, bins=25)
axes[1].set_title("After Box-Cox Transformation")
axes[1].set_xlabel("Transformed Value")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()


## 10. Z-score and Probability Above ₹5,000


In [ ]:
mean_amount = amounts.mean()
std_amount = amounts.std(ddof=1)
threshold = 5000

z_5000 = (threshold - mean_amount) / std_amount
normal_probability = stats.norm.sf(z_5000)
empirical_probability = (amounts > threshold).mean()
lognormal_probability = stats.lognorm.sf(
    threshold, ln_shape, loc=ln_loc, scale=ln_scale
)

print(f"Mean transaction amount: ₹{mean_amount:,.2f}")
print(f"Sample standard deviation: ₹{std_amount:,.2f}")
print(f"Z-score for ₹5,000: {z_5000:.4f}")
print(f"Normal-based P(X > ₹5,000): {normal_probability:.4%}")
print(f"Empirical P(X > ₹5,000): {empirical_probability:.4%}")
print(f"Log-normal fitted P(X > ₹5,000): {lognormal_probability:.4%}")
print(f"Observed transactions above ₹5,000: {(amounts > threshold).sum()} of {len(amounts)}")


## 11. PDF and CDF


In [ ]:
x_grid = np.linspace(amounts.min(), amounts.max(), 500)
pdf_values = stats.lognorm.pdf(x_grid, ln_shape, loc=ln_loc, scale=ln_scale)
cdf_values = stats.lognorm.cdf(x_grid, ln_shape, loc=ln_loc, scale=ln_scale)

fig, axes = plt.subplots(1, 2, figsize=(13,5))

axes[0].hist(amounts, bins=25, density=True, alpha=0.45, label="Observed")
axes[0].plot(x_grid, pdf_values, linewidth=2, label="Log-Normal PDF")
axes[0].set_title("Transaction Amount – PDF")
axes[0].set_xlabel("Transaction Amount (₹)")
axes[0].set_ylabel("Density")
axes[0].legend()

axes[1].plot(x_grid, cdf_values, linewidth=2)
axes[1].set_title("Transaction Amount – CDF")
axes[1].set_xlabel("Transaction Amount (₹)")
axes[1].set_ylabel("Cumulative Probability")
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()


## 12. Final Distribution Analysis


In [ ]:
# Compact final results table
results = pd.DataFrame({
    "Metric": [
        "Number of records",
        "Mean transaction amount",
        "Transaction amount skewness",
        "Bernoulli success probability",
        "Poisson daily lambda",
        "Log-normal K-S statistic",
        "Log-normal K-S p-value",
        "Pareto K-S statistic",
        "Pareto K-S p-value",
        "Box-Cox lambda",
        "Z-score for ₹5,000",
        "Normal P(X > ₹5,000)",
        "Empirical P(X > ₹5,000)",
        "Log-normal P(X > ₹5,000)"
    ],
    "Value": [
        len(df),
        f"₹{mean_amount:,.2f}",
        f"{amounts.skew():.4f}",
        f"{p_success:.4f}",
        f"{lambda_poisson:.4f}",
        f"{ln_ks.statistic:.4f}",
        f"{ln_ks.pvalue:.6f}",
        f"{pa_ks.statistic:.4f}",
        f"{pa_ks.pvalue:.6g}",
        f"{boxcox_lambda:.4f}",
        f"{z_5000:.4f}",
        f"{normal_probability:.4%}",
        f"{empirical_probability:.4%}",
        f"{lognormal_probability:.4%}"
    ]
})
display(results)
